[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/08_Reasoning_and_Inference.ipynb)

# 推理与推断(Reasoning and Inference)

## 概述

使用 **Semantica 推理模块**构建知识图谱、定义规则、执行前向/后向链推理,并为 AI 推理生成解释。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/reasoning/)

## 安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

## 工作流:构建知识图谱 → 定义规则 → 前向/后向链 → 生成解释

In [ ]:
!pip install -qU semantica


In [ ]:
from semantica.kg import GraphBuilder
from semantica.reasoning import Reasoner, ExplanationGenerator


## 步骤 1:构建知识图谱

In [ ]:
builder = GraphBuilder()

entities = [
    {"id": "alice", "type": "Person", "name": "Alice"},
    {"id": "bob", "type": "Person", "name": "Bob"},
    {"id": "charlie", "type": "Person", "name": "Charlie"},
    {"id": "sf", "type": "Location", "name": "San Francisco"},
    {"id": "california", "type": "Location", "name": "California"},
]

relationships = [
    {"source": "alice", "target": "bob", "type": "parent_of"},
    {"source": "bob", "target": "charlie", "type": "parent_of"},
    {"source": "sf", "target": "california", "type": "located_in"},
    {"source": "alice", "target": "sf", "type": "lives_in"},
]

knowledge_graph = builder.build([{"entities": entities, "relationships": relationships}])


## 步骤 2:定义规则

In [ ]:
# Initialize Reasoner
reasoner = Reasoner()

# Define rules using logic syntax
rules = [
    "IF parent_of(?a, ?b) AND parent_of(?b, ?c) THEN grandparent_of(?a, ?c)",
    "IF lives_in(?x, ?y) AND located_in(?y, ?z) THEN lives_in(?x, ?z)"
]

for rule in rules:
    reasoner.add_rule(rule)


## 步骤 3:前向链推理

In [ ]:
# Perform forward chaining to derive new facts
# The Reasoner can infer facts directly from the knowledge graph or a list of facts
inferred_facts = reasoner.infer_facts(knowledge_graph)

print(f"Inferred {len(inferred_facts)} new facts:")
for fact in inferred_facts:
    print(f" - {fact}")


## 步骤 4:后向链推理

In [ ]:
# Define a goal to prove
goal = "grandparent_of(alice, charlie)"

# Perform backward chaining
proof = reasoner.backward_chain(goal)

if proof:
    print(f"Goal '{goal}' proven successfully!")
else:
    print(f"Could not prove goal '{goal}'.")


## 步骤 5:生成解释

In [ ]:
generator = ExplanationGenerator()

# If we have a proof from backward chaining, explain it
if proof:
    proof_explanation = generator.generate_explanation(proof)
    print("Explanation for backward chaining proof:")
    print(proof_explanation.natural_language)


## 总结

推理与推断工作流:
- 已构建知识图谱
- 已定义推理规则
- 已将事实加载到引擎
- 已执行前向链推理
- 已执行后向链推理
- 已生成解释

---

### 深入探讨:推理模块

本节深入介绍 Semantica 的推理能力。通过稳健、可复现的示例,学习规则语法、事实格式、链式推理策略和解释生成。

**你将练习**
- 使用变量和谓词定义规则
- 以谓词形式加载事实
- 运行前向和后向链推理
- 生成人类可读的解释

In [ ]:
from semantica.kg import GraphBuilder
from semantica.reasoning import Reasoner, ExplanationGenerator

builder = GraphBuilder()
reasoner = Reasoner()
explainer = ExplanationGenerator()


### 规则语法

规则使用谓词逻辑,变量以 `?` 为前缀。

- 示例:`IF parent_of(?a, ?b) AND parent_of(?b, ?c) THEN grandparent_of(?a, ?c)`
- 同一规则中,变量会在各谓词间合一(unify)
- 当条件匹配时,结论会作为新事实被加入

In [ ]:
entities = [
    {"id": "alice", "type": "Person", "name": "Alice"},
    {"id": "bob", "type": "Person", "name": "Bob"},
    {"id": "charlie", "type": "Person", "name": "Charlie"},
    {"id": "sf", "type": "Location", "name": "San Francisco"},
    {"id": "california", "type": "Location", "name": "California"}
]

relationships = [
    {"source": "alice", "target": "bob", "type": "parent_of"},
    {"source": "bob", "target": "charlie", "type": "parent_of"},
    {"source": "sf", "target": "california", "type": "located_in"},
    {"source": "alice", "target": "sf", "type": "lives_in"}
]

knowledge_graph = builder.build([{"entities": entities, "relationships": relationships}])
print(len(knowledge_graph.get("entities", [])))
print(len(knowledge_graph.get("relationships", [])))


In [ ]:
rules = [
    "IF parent_of(?a, ?b) AND parent_of(?b, ?c) THEN grandparent_of(?a, ?c)",
    "IF lives_in(?x, ?y) AND located_in(?y, ?z) THEN lives_in(?x, ?z)"
]
for r in rules:
    reasoner.add_rule(r)


In [ ]:
for rel in relationships:
    fact = f"{rel['type']}({rel['source']}, {rel['target']})"
    reasoner.add_fact(fact)

derived = reasoner.forward_chain()
print(len(derived))
for d in derived:
    print(d.conclusion)


In [ ]:
goals = [
    "grandparent_of(alice, charlie)",
    "lives_in(alice, california)"
]
for g in goals:
    proof = reasoner.backward_chain(g)
    print(g)
    print(bool(proof))


In [ ]:
if derived:
    exp = explainer.generate_explanation(derived[0])
    print(exp.natural_language)

goal = "grandparent_of(alice, charlie)"
proof = reasoner.backward_chain(goal)
if proof:
    pexp = explainer.generate_explanation(proof)
    print(pexp.natural_language)
